In [1]:
from pathlib import Path
import torch
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader

In [2]:
DATA_DIR= Path('../data/raw/')

print(DATA_DIR.exists())
print(DATA_DIR.resolve())

True
C:\Users\drago\Desktop\AnimalProject\data\raw


In [3]:
dataset= datasets.ImageFolder(root=DATA_DIR)

print('NR of pictures:', len(dataset))
print('NR of classes:', len(dataset.classes))
classes= [a for a in dataset.classes]
print('classes:', classes)


NR of pictures: 26179
NR of classes: 10
classes: ['butterfly', 'cat', 'chicken', 'cow', 'dog', 'elephant', 'horse', 'sheep', 'spider', 'squirrel']


In [4]:
index= []
labels= []

for i, sample in enumerate(dataset.samples):
    image_path, label= sample

    index.append(i)
    labels.append(label)


print(index[0], labels[0])
print('antal index:',len(index), 'antal labels:',len(labels))

print('antal bilder:',len(dataset.samples))
print('EX:', dataset.samples[0])

0 0
antal index: 26179 antal labels: 26179
antal bilder: 26179
EX: ('..\\data\\raw\\butterfly\\OIP--04ndbWy7I04gsPgu9qOeQHaHs.jpeg', 0)


In [5]:
X_train, X_temp, y_train, y_temp= train_test_split(
    index, 
    labels,
    stratify=labels,
    test_size=0.3,
    random_state=0
)

X_valid, X_test, y_valid, y_test= train_test_split(
    X_temp,
    y_temp,
    stratify= y_temp,
    test_size=0.5,
    random_state=0
)

print(X_train[:5], y_train[:5])

[22903, 8348, 12201, 22944, 2849] [8, 3, 4, 8, 1]


In [6]:
train_data= Subset(dataset, X_train)
valid_data= Subset(dataset, X_valid)
test_data= Subset(dataset, X_test)

print('Tränings data:', len(train_data))
print('Validerings data:', len(valid_data))
print('Test data:', len(test_data))

Tränings data: 18325
Validerings data: 3927
Test data: 3927


In [7]:
train_transform= transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
valid_transform= transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [8]:
train_data_full= datasets.ImageFolder(
    DATA_DIR,
    train_transform
)
valid_data_full= datasets.ImageFolder(
    DATA_DIR,
    valid_transform
)
test_data_full= datasets.ImageFolder(
    DATA_DIR,
    valid_transform
)



In [9]:
train_data= Subset(train_data_full, X_train)
valid_data= Subset(valid_data_full, X_valid)
test_data= Subset(test_data_full, X_test)

image, label = train_data[0]

print(type(image))
print(image.shape)
print(label)
print(train_data_full.classes[label])

<class 'torch.Tensor'>
torch.Size([3, 224, 224])
8
spider


In [10]:
BATCH_SIZE=32

train_data_loader= DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)
valid_data_loader= DataLoader(
    dataset=valid_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)
test_data_loader= DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [11]:
images, labels= next(iter(train_data_loader))

print('Första batch shape:', images.shape)
print('Första 10 labels:', labels[:10])

Första batch shape: torch.Size([32, 3, 224, 224])
Första 10 labels: tensor([4, 8, 5, 7, 9, 8, 6, 4, 3, 0])


In [12]:
import torchvision.models as models
import torch.nn as nn

model = models.resnet18(weights='DEFAULT')

for param in model.parameters():
    param.requires_grad = False


model.fc = nn.Linear(model.fc.in_features, 10)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print('Använder:', device)

Använder: cuda


In [13]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [14]:
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    # TRÄNING
    model.train()
    train_loss = 0
    correct = 0

    for images, labels in train_data_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = correct / len(train_data)

    # VALIDERING
    model.eval()
    val_loss = 0
    val_correct = 0

    with torch.no_grad():
        for images, labels in valid_data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc = val_correct / len(valid_data)

    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | '
          f'Train loss: {train_loss/len(train_data_loader):.3f} | '
          f'Train acc: {train_acc:.3f} | '
          f'Val loss: {val_loss/len(valid_data_loader):.3f} | '
          f'Val acc: {val_acc:.3f}')

Epoch 1/5 | Train loss: 0.672 | Train acc: 0.799 | Val loss: 0.208 | Val acc: 0.941
Epoch 2/5 | Train loss: 0.432 | Train acc: 0.857 | Val loss: 0.191 | Val acc: 0.941
Epoch 3/5 | Train loss: 0.419 | Train acc: 0.860 | Val loss: 0.169 | Val acc: 0.944
Epoch 4/5 | Train loss: 0.406 | Train acc: 0.864 | Val loss: 0.172 | Val acc: 0.944
Epoch 5/5 | Train loss: 0.397 | Train acc: 0.867 | Val loss: 0.163 | Val acc: 0.947


In [15]:
torch.save(model.state_dict(), '../models/resnet18_animals.pth')
print('Modell sparad!')

Modell sparad!
